In [1]:
from tqdm import tqdm
import pickle
import glob
import numpy as np
from tqdm import tqdm
import pickle
import glob
from experiments.end_to_end.proof_node import ErrorNode, Status
import math

bestfs_path = "../runs/internlm/internlm/2025_02_14/16_25_13/traces/0/"
cg_path = "../runs/internlm/internlm_cg/2025_02_18/12_00_59/traces/0/"
dpp_path = "../runs/internlm/dpp_critic/2025_02_24/16_13_55/traces/0/"


def load_traces(path):
    files = glob.glob(path + '*', recursive=True)
    traces = []

    for file in tqdm(files):
        traces.append(pickle.load(open(file, "rb")))

    return traces

In [2]:


# bfs_traces = load_traces(bfs_path)
bestfs_traces = load_traces(bestfs_path)
cg_traces = load_traces(cg_path)
dpp_traces = load_traces(dpp_path)

goal_scores = []
for trace in dpp_traces:
    goal_scores.extend(list(set([e.goal_logprob for e in trace.trace])))


100%|██████████| 244/244 [00:02<00:00, 87.57it/s] 


In [10]:
list(dpp_traces[0].nodes.values())[0].out_edges


[Edge(tactic='norm_num [Finset.sum_range_succ, Finset.sum_range_succ', tac_logprob=-1.428685087328148, goal_logprob=-1.2119140625, time=0.06920495699159801),
 Edge(tactic='norm_num [Finset.sum_range_succ, Finset.sum_range_succ', tac_logprob=-1.428685087328148, goal_logprob=-1.2119140625, time=0.06920662400079891),
 Edge(tactic='norm_num [Finset.sum_range_succ, Finset.sum_range_succ', tac_logprob=-1.428685087328148, goal_logprob=-1.2119140625, time=0.06921414899989031),
 Edge(tactic='norm_num [Finset.sum_range_succ, Finset.sum_range_succ', tac_logprob=-1.428685087328148, goal_logprob=-1.2119140625, time=0.06924687100399751),
 Edge(tactic='norm_num [Finset.sum_range_succ, Finset.sum_range_succ', tac_logprob=-1.428685087328148, goal_logprob=-1.2119140625, time=0.07030770900018979),
 Edge(tactic='norm_num [Finset.sum_range_succ, Finset.sum_range_succ', tac_logprob=-1.428685087328148, goal_logprob=-1.2119140625, time=0.06922425300581381),
 Edge(tactic='norm_num [Finset.sum_range_succ, Finse

In [2]:

from vllm import LLM

model = LLM(model='intfloat/e5-small-v2')

INFO 02-26 14:09:05 __init__.py:183] Automatically detected platform cuda.
INFO 02-26 14:09:07 config.py:2314] Downcasting torch.float32 to torch.float16.
INFO 02-26 14:09:16 config.py:520] This model supports multiple tasks: {'score', 'classify', 'reward', 'embed'}. Defaulting to 'embed'.
INFO 02-26 14:09:16 llm_engine.py:232] Initializing an LLM engine (v0.7.0) with config: model='intfloat/e5-small-v2', speculative_config=None, tokenizer='intfloat/e5-small-v2', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=512, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 02-26 14:09:21 model_runner.py:1115] Loading model weights took 0.0633 GB


In [4]:

tokenizer = model.get_tokenizer()

In [8]:
token_ids = tokenizer.encode('query: x : ℕ\nh₀ : ↑x + 4 / 100 * ↑x = 598\n⊢ 100 * x = 100 * 575' * 100)


Token indices sequence length is longer than the specified maximum sequence length for this model (3001 > 512). Running this sequence through the model will result in indexing errors


In [13]:
tokenizer.encode('query: x : ℕ\nh₀ : ↑x + 4 / 100 * ↑x = 598\n⊢ 100 * x = 100 * 575' * 100, truncate=True)

TypeError: PreTrainedTokenizerFast._batch_encode_plus() got an unexpected keyword argument 'truncate'

In [9]:
len(token_ids)

3001

In [ ]:

scores_ = list(set([(e.src.goal, e.goal_logprob) for e in dpp_traces[18].trace]))

embs = []
tokenizer = model.get_tokenizer()
for goal, _ in tqdm(scores_):
    prompt_token_ids = tokenizer.encode('query: ' + goal)  #, return_tensors="pt")

    # Truncate prompt_token_ids
    prompt_token_ids = prompt_token_ids[:512]

    embs.append(model.encode(prompt_token_ids=prompt_token_ids, use_tqdm=False, )[
                    0].outputs.data)

# normalise scores to be in [0,1]

max_score = max(scores_, key=lambda x: x[1])[1]
min_score = min(scores_, key=lambda x: x[1])[1]
scores_ = [(s[0], (s[1] - min_score + 1) / (max_score - min_score)) for s in scores_]

scores_ = sorted(scores_, key=lambda x: x[1], reverse=True)

print(np.array(embs) @ np.array(embs).T)

from dppy.finite_dpps import FiniteDPP

embs = [e * scores_[i][1] for i, e in enumerate(embs)]

embs = np.array(embs) @ np.array(embs).T

scores_[:10]
inds = sample_conditional_dpp(embs, [0], k=3)
print(max(scores_, key=lambda x: x[1])[1])
[scores_[i] for i in inds]

DPP = FiniteDPP('likelihood', **{'L': embs})
# inds = DPP.sample_exact()
inds = DPP.sample_exact_k_dpp(size=3, mode='KuTa12')  # ,rng=rng)

print(max(scores_, key=lambda x: x[1])[1])
[scores_[i] for i in inds]


from numpy.linalg import inv


# implements eqn 42 from https://arxiv.org/pdf/1207.6083
def sample_conditional_dpp(L, set0, k=None):
    '''
    Wrapper function for the sample_dpp Matlab code written by Alex Kulesza
    Given a kernel matrix L, returns a sample from a k-DPP.
    The code is hacked in a way that if a set A is provied, samples from a conditional
    dpp given A are produced
    L:     kernel matrix
    set:   index of the conditional elements. Integer numpy array containing the locations
            (starting in zero) relative to the rows of L.
    k:     size of the sample from the DPP
    '''
    set0 = np.array(set0)  # matlab starts counting in one
    # Calculate the kernel for the marginal
    Id = np.array([1] * L.shape[0])
    Id[set0] = 0
    Id = np.diag(Id)
    L_compset_full = inv(Id + L)
    L_minor = inv(np.delete(np.delete(L_compset_full, tuple(set0), axis=1), tuple(set0), axis=0))
    L_compset = L_minor - np.diag([1] * L_minor.shape[0])

    DPP = FiniteDPP('likelihood', **{'L': L_compset})

    # inds = DPP.sample_exact()
    sample = DPP.sample_exact_k_dpp(size=k - 1, mode='KuTa12')  # ,rng=rng)

    # Compute the sample

    return set0.tolist() + [s + 1 for s in sample]